### Structured Output

Models can be requested to proveide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structure.

In [34]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.5'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A44AC4AD70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A44AC4B6F0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [35]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(desciption="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

C:\Users\Durga\AppData\Local\Temp\ipykernel_14200\3002463774.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  director:str=Field(desciption="The director of the movie")


In [36]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.5'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A44AC4AD70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A44AC4B6F0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [37]:
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Message output alongside parsed structure

In [38]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(desciption="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

C:\Users\Durga\AppData\Local\Temp\ipykernel_14200\1729200603.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  director:str=Field(desciption="The director of the movie")


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the movie Inception. We can provide info: director, rating, title, year. Could also give more description. Use the function Movie to return structured data. Probably call function with director Christopher Nolan, rating maybe 8.8 (IMDb), title Inception, year 2010.', 'tool_calls': [{'id': 'fc_1cd9177e-6fc7-4244-ad2e-6f2730df7fbe', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 152, 'total_tokens': 268, 'completion_time': 0.242116467, 'completion_tokens_details': {'reasoning_tokens': 64}, 'prompt_time': 0.006840388, 'prompt_tokens_details': None, 'queue_time': 0.33692494, 'total_time': 0.248956855}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason

### Nested Structure

In [39]:
from pydantic.v1 import StrBytes
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=None)

### TypedDict


TypeDict provides a simpler alternative using Python's build-in typing, ideal when you don't need runtime validation. 

In [40]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypeddict = model.with_structured_output(MovieDict)
response=model_withtypeddict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [41]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about movie Bahubali")
response

{'budget': 28000000,
 'cast': [{'name': 'Prabhas', 'role': 'Shivudu / Mahendra Baahubali'},
  {'name': 'Rana Daggubati', 'role': 'Bhallaladeva'},
  {'name': 'Anushka Shetty', 'role': 'Devasena'},
  {'name': 'Tamannaah Bhatia', 'role': 'Avanthika'},
  {'name': 'Sathyaraj', 'role': 'Kattappa'},
  {'name': 'Ramya Krishna', 'role': 'Sivagami'},
  {'name': 'Nassar', 'role': 'Kumara Varma'},
  {'name': 'Mohan Babu', 'role': 'Bijjaladeva'}],
 'genres': ['Action', 'Adventure', 'Drama', 'Fantasy'],
 'title': 'Baahubali: The Beginning',
 'year': 2015}

In [42]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there are not really any restrictions. You create it using the @dataclass decorator

In [43]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [44]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(desciption="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages":[{"role":"user", "content":"Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])


C:\Users\Durga\AppData\Local\Temp\ipykernel_14200\3577296297.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desciption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  name: str = Field(desciption="The name of the person")


OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [ ]:
# Using TypedDict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str 
    email: str
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo #Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content":"Extract info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

TypeError: 'module' object is not iterable

In [45]:
## DataClass
from langchain.agents import create_agent
from dataclasses import dataclass

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str 
    email: str
    phone: str

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo #Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content":"Extract info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}